<a id="unit-u.5a8b4855f062494299248f8927288d9b"></a>

<a id="compression-materialized-views-and-inverted-indexes"></a>

# Сығу, материалданған көріністер және кері индекстер

**6-модуль: Кесте қасиеттері, материалданған көріністер және индекстер** · [Сабақ](course.md) · [Тест](quiz06_properties_mv_and_indexes.ipynb)

3-модульде қатынас журналынан сәтсіз checkout іздедіңіз, 4-модульде әр сервис пен минут сұраулары мен ақауларын өз кестеңізде сақтадыңыз. Бұл жұмыста Doris ішінде оны жылдам әрі үнемді ететінді қарайсыз: экспорттаушының `otel_logs` қасиеттері, қатынас журналы іздеуіне жауап беретін кері индекс, Doris 4-модуль санақтарымен өзекті ұстайтын материалданған көрініс және телеметрия диск көлемін анықтайтын сығу. Әр ұяшықты қайта орындауға болады.

Шамамен 40 минут жоспарлаңыз. Төлем ақауы қосылған 1-модульдің DOG стегі мен демосы қажет (1-зертханалық жұмыстың 5-бөлімі).

<a id="unit-u.1f7f639440ef44ec8bea32797f57b6f8"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты инициализациялау

`lab.shell(...)` не `lab.sql(...)` қолданбас бұрын осы ұяшықты бір рет орындаңыз.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


<a id="unit-u.c5315c6780364486a52d675911da2447"></a>

<a id="1-check-the-failed-checkouts-in-the-access-log"></a>

## 1. Қатынас журналындағы сәтсіз checkout-ты тексеру

Бұл жұмыстың іздеуі — 3-модульдегісі: дүкен проксиінің сәтсіз checkout қатынас журналы жолдары `"POST /api/checkout ... HTTP/1.1" 500`. Оларды санаңыз:

In [ ]:
failed_checkouts = lab.wait_for_rows(
    """
    SELECT COUNT(*) AS failed_checkouts, MAX(timestamp) AS latest
    FROM otel_logs
    WHERE body MATCH_PHRASE 'POST /api/checkout'
      AND body MATCH_PHRASE 'HTTP/1.1 500'
    HAVING COUNT(*) > 0
    """,
    title="Access-log lines of failed checkouts in otel_logs",
    hint="No checkout has failed. Check that the demo runs (lab.start_demo() in Module 1) and that paymentFailure "
         "is 50% (section 5 of Lab 1), wait a minute or two, then rerun this cell.",
    timeout_seconds=180,
)

<a id="unit-u.bf823707b16649c0b0a708ef7be20803"></a>

**Күтілетін нәтиже**

Бір жол: жалаушаны қосқалы әр сәтсіз тапсырысқа бір қатынас журналы жолы және ең соңғысының UTC уақыты.

<a id="unit-u.952da040ffc34997887329127e625a24"></a>

<a id="2-read-the-tables-definition"></a>

## 2. Кесте анықтамасын оқу

`SHOW CREATE TABLE` кестені қазіргі күйінде барлық қасиетімен құратын команданы қайтарады. `lab.show_create_table(...)` оны мәтінмен көрсетеді:

In [ ]:
lab.show_create_table("otel_logs");

<a id="unit-u.f0c8e48f871a4f70bdfc9e5d7b41a499"></a>

**Күтілетін нәтиже**

`otel_logs` командасы, шамамен 60 жол. Жоғарыдан оқыңыз:

- **Бағандар.** `body` — `text`. `resource_attributes` пен `log_attributes` — `variant`: Doris әр атрибут кілтін `variant_max_subcolumns_count` = 2048 шегіне дейін жеке бағанмен сақтайды, сондықтан бір кілт сұрауы тек соны оқиды.
- **Индекстер.** Әр бағанға бір `USING INVERTED` индексі. Тек `idx_body` қасиеттері бар: `"parser" = "unicode"` әр хабарды сөздерге бөледі, `"lower_case" = "true"` кіші әріппен индекстейді, `"support_phrase" = "true"` фразалық іздеу үшін әр сөз орнын да тіркейді. 3-бөлім соны өлшейді.
- **Кілт, бөлімдер, bucket-тер.** 4 және 5-модульдердегі `DUPLICATE KEY(timestamp, service_name)`, күндік бөлімдер және `RANDOM BUCKETS AUTO`.
- **`compaction_policy` = `time_series`** және `time_series_compaction_*` баптаулары. Әр жүктеу тиетін әр tablet-ке шағын файлдар жиынын, rowset, жазады; compaction оларды фонда біріктіреді. Әдепкі саясат ұсақ rowset-терді деңгейлеп ірілендіріп, кей деректі көп рет қайта жазады. Уақыт қатары саясаты уақыт ретімен тек қосылатын дерекке сай: tablet rowset-тері шамамен `time_series_compaction_goal_size_mbytes` (1024 MB), не файл саны, не бір сағат шегіне жеткенше күтіп, бір рет соңғы rowset-ке біріктіреді. Осындай көп жүктеу алатын кестелерде диск пен CPU үнемдейді.
- **`group_commit_interval_ms` және `group_commit_data_bytes`.** Тек Group Commit қолданатын жүктеулерге қатысты; ол көп шағын жүктеуді бір транзакцияға, мұнда 10 секунд не 128 MB бойынша жинайды. Жүктеу тақырыппен сұрайды не кесте анықтамада көрінер `group_commit_mode` арқылы әдепкі етеді. Экспорттаушы екеуін де істемейді: `batch` өңдегіші ірі ұстайтын әр пакетті бір Stream Load ретінде жібереді.
- **`compression` жоқ.** Экспорттаушы кестені `"compression" = "zstd"` арқылы құрған. `SHOW CREATE TABLE` әдепкі қасиеттерді көрсетпейді, ал ZSTD — Doris 4.x әдепкісі. 5-бөлім LZ4-пен салыстырады.

<a id="unit-u.0f8972f3230e46658681562717b5a05b"></a>

<a id="3-measure-what-the-index-of-body-does-for-the-search"></a>

## 3. `body` индексінің іздеуге әсерін өлшеу

3-модульде Discover `body:"POST /api/checkout" AND body:"HTTP/1.1 500"` іздеуін екі `MATCH_PHRASE` шартына айналдырды. `idx_body` арқылы Doris әр фразаның сөздерін индекстен іздеп, тек сөздері ретімен тұрған жолдарды оқиды. Бұл қанша оқуды үнемдейді? Query Profile көрсетеді. Кесте сканерлеуі сияқты әр оператор үшін Doris не істегенін, оның ішінде қанша жолды және неге өткізгенін тіркейді. `enable_profile` айнымалысы `true` болған сеанстың әр сұрауына профиль жинайды. Келесі ұяшық оны осы ноутбук сеансына қатысты тағы үш айнымалымен орнатады:

- `profile_level = 2`: профиль сканерлеу жол санауыштарын қамтиды. Doris 4.1 әдепкісі осы; 1-деңгей көбін көрсетпейді.
- `enable_sql_cache = false`: кесте біраз өзгермесе, SQL кэші қайталанған сұрау нәтижесін сканерлемей бере алады, сонда профильде сканерлеу жоқ.
- `enable_condition_cache = false`: шарт кэші сүзгінің бұрынғы орындалуында деректің қай бөліктерінде сәйкестік болмағанын есте сақтап, қайталауда өткізеді. Сондықтан қайталау санауыштары индекс сүзген барлық жолды көрсетпейді.

In [ ]:
lab.execute("""
SET enable_profile = true,
    profile_level = 2,
    enable_sql_cache = false,
    enable_condition_cache = false
""", title="Collect Query Profiles in this session, and scan the table on every run");

<a id="unit-u.ed28f77f11734a648feb2b08d1f4755d"></a>

**Күтілетін нәтиже**

Карточка команданың сәттілігін айтады. Айнымалылар ядро қайта іске қосылғанша ноутбуктің Doris қосылымына қолданылады; қайта іске қосқанда осы жұмыстың алғашқы ұяшығын және осыны қайталаңыз. MySQL клиентінде сұраулардан бұрын сол `SET` командасын орындаңыз.

<a id="unit-u.d28e55644a354b09894e30d43145c8e2"></a>

Сәтсіз checkout-ты сервиспен санап, сұрау профилін алыңыз. `lab.profile_query(...)` сұрауды орындап, профилін үш қадаммен табады; оларды MySQL клиенті мен терминалда да орындай аласыз:

1. `SELECT last_query_id()` сеанстың соңғы сұрау ID-ін береді. Ол профильдің де ID-і.
2. `SHOW QUERY PROFILE` әрқайсысының ID-і мен SQL-і бар жаңа профильдерді тізеді. Сұрау ID-і тізімде болса, Doris профилін жинаған.
3. FE HTTP API профиль мәтінін ID бойынша қайтарады. Ұяшық сұрауды `curl` командасы ретінде көрсетеді.

FE веб-интерфейсі сол профильдерді тізеді: http://127.0.0.1:28030 ашыңыз (`DORIS_OBS_HTTP_PORT` порты), құпиясөзсіз `root` болып кіріп, QueryProfile ашыңыз.

Сұрау жай санаудың орнына сервиспен топтайды: тек `COUNT(*)` болса, Doris жол оқымай индекстен жауап бере алады (жоспарда `COUNT_ON_INDEX`), санауыштар шынымен оқылған жолдарды көрсетпес еді.

In [ ]:
INDEX_QUERY = """
SELECT service_name, COUNT(*) AS failed_checkouts
FROM otel_logs
WHERE body MATCH_PHRASE 'POST /api/checkout'
  AND body MATCH_PHRASE 'HTTP/1.1 500'
GROUP BY service_name
"""
profile = lab.profile_query(INDEX_QUERY, title="Failed checkouts in the access log, by service")

<a id="unit-u.4078d4cd0c1a4147902698b1dc3894a2"></a>

**Күтілетін нәтиже**

Алдымен нәтиже: бірінші ұяшық тапқандай не сәл көбірек сәтсіз checkout саны бар бір `frontend-proxy` жолы.

Кейін үш қадам: сұрау ID-і, `SHOW QUERY PROFILE` ішіндегі басталу уақыты мен ұзақтығы (`Total`, бірнеше миллисекунд) бар жолы, `curl` командасы; соңынан жиналған өрісте бірнеше мың жолдық профиль мәтіні. Мәтін сұраудың `Summary` бөлімінен басталады; `ChangedSessionVariables` `SET` ұяшығы әдепкіден өзгерткен айнымалыларды тізеді. Кейін жоспар операторлары екі рет беріледі. Doris әр оператордың бірнеше данасын параллель орындайды: `MergedProfile` даналарды біріктіреді, `DetailProfile` әрқайсысын бөлек тізеді. Кесте сканерлеуінің жол санауыштары `DetailProfile` ішіндегі әр `OLAP_SCAN_OPERATOR` астындағы `SegmentIterator` ішінде: мәтіннен `RowsInvertedIndexFiltered` іздеңіз.

<a id="unit-u.5f28b26e79364221beff7311e54b7b4b"></a>

Сканерлеу қарастырған әр кесте жолы оны алып тастаған қадаммен не өткен жол ретінде бір рет саналады: zone map өткізді, кері индекс өткізді, сүзгі оқып тастады немесе оқып қалдырды. Сканерлеу даналары бойынша осы санауыштарды қосыңыз:

In [ ]:
with_index = lab.scan_filters(profile, title="How the scan of otel_logs filtered its rows")
assert with_index["RowsInvertedIndexFiltered"] > 0, (
    "The inverted index filtered no rows. Check that the first cell of this Lab found failed checkouts, "
    "then rerun the query and the cells after it."
)

<a id="unit-u.5e037cd31c8d4210828613874a6ab76d"></a>

**Күтілетін нәтиже**

Қосындысы сұрау кезіндегі `otel_logs` жол санына тең бес санауыш:

- `RowsStatsFiltered`: 0 не бірнеше жол. Doris жолдарды сегменттерде (дерек файлдарында) сақтап, әр сегмент пен оның әр жол беті үшін бағанның ең кіші/үлкен мәндерін, zone map-терді, ұстайды. Олар мән аралығын жоққа шығарады, мәтін ішіндегі сөздерді емес, сондықтан бұл іздеуде ештеңе өткізбейді.
- `RowsInvertedIndexFiltered`: кестенің дерлік барлық, ондаған мың жолы. Индекс екі фраза сөздері ретімен бар жолдарды тізеді; Doris дерек оқымай тұрып басқаларын алып тастайды.
- `RowsVectorPredFiltered` және `RowsExprPredFiltered`: 0. Бұлар Doris оқып, сүзгімен салыстырғаннан кейін тастайтын жолдар болар еді; индекспен тек сәйкес жолдар оқылады.
- `RowsRead`: сұрау санағандай сәтсіз checkout саны.

<a id="unit-u.e104c910af0242178c3ce498cf2178da"></a>

Енді сол сұрауды индекссіз өлшеңіз. `enable_inverted_index_query = false` кезінде Doris сүзгілерді кері индекссіз бағалайды: әр жолдың `body` мәтінін оқып, сөздерге бөліп, фразалармен салыстырады. Келесі ұяшық осы сеанста индексті өшіріп, сұрауды орындайды, профиль санауыштарын қосады және индексті қайта қосады:

In [ ]:
import pandas as pd

lab.execute("SET enable_inverted_index_query = false", title="Stop using inverted indexes in this session")
try:
    profile_without_index = lab.profile_query(INDEX_QUERY, title="Failed checkouts by service, without the index")
    without_index = lab.scan_filters(profile_without_index, title="How the scan filtered the rows without the index")
finally:
    # Use the index again, also when a step above failed.
    lab.execute("SET enable_inverted_index_query = true", title="Use inverted indexes again")

pd.DataFrame({"with the index": with_index, "without the index": without_index})

<a id="unit-u.e32d8b56d4d44215b9529e2a979aee5f"></a>

**Күтілетін нәтиже**

Сұрау бұрынғы санақты және арада келген ақауларды қайтарады. Соңғы кесте екі өлшемді қатар көрсетеді:

- `RowsInvertedIndexFiltered` — 0, орнына шамамен сонша жол `RowsExprPredFiltered` астында. `MATCH_PHRASE` — сүзгі өрнегі: Doris сол жолдардың `body` мәтінін оқып, токендерге бөліп, сәйкес емесін тапты.
- `RowsRead` бірдей: сол жолдар сәйкеседі.

Индекссіз Doris бірнеше сәтсіз checkout табу үшін әр хабарды, индекспен тек соларды оқыды. Мұндай шағын кестеде екі сұрау да миллисекунд алады; миллиардтаған журнал жолында әр хабарды оқу мен токендеу әр іздеуде диск пен CPU жұмсайды. `LIKE '%...%'` әр хабарды солай оқиды, сондықтан іздеу индексті бағанда `MATCH_ANY`, `MATCH_ALL`, `MATCH_PHRASE` қолданады.

`RowsInvertedIndexFiltered` осы сұраудың осы сканерлеуінде индекс не істегенін дәлелдейді: Doris оқымай тұрып осы жолдарды алып тастады. Жеке өзі сұрау қаншалықты жылдамдағанын, басқа неге уақыт жұмсағанын не басқа сүзгі индекс қолдана алатынын көрсетпейді. Ондай тұжырым жасайтын әр сұрауды өлшеңіз.

<a id="unit-u.5cf7900a6db24d4cb8cc2966a8ca52e5"></a>

<a id="4-a-materialized-view-of-the-incidents-counts"></a>

## 4. Инцидент санақтарының материалданған көрінісі

4-модульде әр сервис пен минут сұраулары мен сәтсіздерін `requests_per_minute` ішіне `INSERT` арқылы жүктеп, тапсырма әр span-ды дәл бір рет жүктеуі керегін көрдіңіз. **Материалданған көрініс** сол қорытындыны жүктеу тапсырмасынсыз сақтайды. Синхронды көрініс бір кестеге тиесілі: құрғанда Doris кесте жолдарынан есептейді, кейін әр жүктеумен бір транзакцияда жаңартады. Сұраулар кестені атай береді; жауап бере алса, Doris көріністі оқиды.

`otel_traces` үстінде 4-модуль санақтарының көрінісін құрыңыз. Doris 4.1 кесте бағанымен атауы бірдей бағаны бар көріністі қабылдамайды, сондықтан әр баған `mv_` деп басталатын жеке атау алады:

In [ ]:
created_after = lab.sql("SELECT NOW() AS doris_time", title="The time on Doris before the view is created")["doris_time"][0]
lab.execute("DROP MATERIALIZED VIEW IF EXISTS server_minutes_mv ON otel_traces",
            title="Drop the view if an earlier run created it, so that the cell can be rerun")
lab.execute("""
CREATE MATERIALIZED VIEW server_minutes_mv AS
SELECT date_trunc(end_time, 'minute') AS mv_request_minute,
       service_name AS mv_service_name,
       COUNT(*) AS mv_requests,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS mv_failed_requests
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER'
GROUP BY date_trunc(end_time, 'minute'), service_name
""", title="Create the materialized view server_minutes_mv on otel_traces")
lab.wait_for_rows(
    f"""
    SHOW ALTER TABLE MATERIALIZED VIEW FROM otel
    WHERE TableName = 'otel_traces' AND RollupIndexName = 'server_minutes_mv'
      AND State = 'FINISHED' AND CreateTime >= '{created_after}'
    """,
    title="The job that builds server_minutes_mv, once it has finished",
    hint="Doris has not finished building the view. Check its State and Msg with SHOW ALTER TABLE "
         "MATERIALIZED VIEW FROM otel, then rerun this cell.",
);

<a id="unit-u.ee77acf21bf34e238d65f385016ffeae"></a>

**Күтілетін нәтиже**

Doris уақыты, командалар сәтті деген екі карточка, кейін `SHOW ALTER TABLE MATERIALIZED VIEW` ішінде `State` мәні `FINISHED` бір жол; әдетте бірнеше секундта. `CREATE MATERIALIZED VIEW` Doris қабылдаған бойда қайтады: кейін тапсырма кестеде бар жолдардан көрініс құрады, тапсырма біткеннен кейін ғана көрініс жауап береді. Жаңа көрініске сүйенбес бұрын тапсырмасын тексеріңіз. `SHOW ALTER TABLE MATERIALIZED VIEW` бұрынғы көрініс тапсырмаларын да тізеді, сондықтан ұяшық тек белгіленген уақыттан кейін құрылғандарын сұрайды.

Ұяшық алдымен көріністі жояды, өйткені Doris 4.1.3 `IF NOT EXISTS` болса да бар көріністі өткізбейді: бар екенін не баған атаулары алынған болса `Duplicate column name` деп хабарлайды. Сондықтан қайталау көріністі қайта құрады.

<a id="unit-u.947292c9db9646908d6f547bec51b96c"></a>

Көрініс кестемен бірге оның индекстерінің бірі ретінде сақталады. `DESC ... ALL` кесте мен әр көрінісінің бағандарын тізеді:

In [ ]:
lab.sql("DESC otel_traces ALL", title="The columns of otel_traces and of its materialized view",
        columns=["IndexName", "IndexKeysType", "Field", "Type", "Extra", "DefineExpr", "WhereClause"])
lab.sql("""
SELECT (SELECT COUNT(*) FROM otel_traces WHERE span_kind = 'SPAN_KIND_SERVER') AS server_spans,
       (SELECT COUNT(*) FROM otel_traces INDEX server_minutes_mv) AS index_hint_rows,
       (SELECT COUNT(*) FROM (
           SELECT date_trunc(end_time, 'minute'), service_name
           FROM otel_traces
           WHERE span_kind = 'SPAN_KIND_SERVER'
           GROUP BY date_trunc(end_time, 'minute'), service_name
       ) AS groups_by_service_minute) AS service_minute_groups
""", title="Server spans, index-hint row count, and global service/minute groups");

<a id="unit-u.c3ecabb5d3264c27b9b3da2dc3e25c2c"></a>

**Күтілетін нәтиже**

Бірінші кесте `otel_traces` 19 бағанын, кейін көріністі тізеді: `IndexName` — `server_minutes_mv`, `IndexKeysType` — `AGG_KEYS`, төрт баған. `mv_request_minute` пен `mv_service_name` — кілт; `mv_requests` пен `mv_failed_requests` үшін `Extra` ішінде `SUM`, ал `DefineExpr` әр өрнекті көріністің `WHERE` шартымен береді. Көрініс — кестенің AGGREGATE индексі: 4-модульдегі `requests_per_minute` сияқты `SUM` арқылы біріктіреді, бірақ Doris өзі толтырады.

Екінші кесте бір командадан үш санақ береді. `server_spans` бастапқы сервер span-дарын санайды. `index_hint_rows` — `INDEX server_minutes_mv` арқылы агрегат индексін оқығандағы санақ; ол жалпы сервис/минут топ санынан артық болуы мүмкін, әр топқа бір соңғы жол деп түсінуге болмайды. `service_minute_groups` деректі нақты топтап, сол жалпы топтарды санайды. Есеп түйіршіктілігін сипаттау үшін осы соңғы бағанды қолданыңыз.

`SELECT ... FROM otel_traces INDEX server_minutes_mv` индексті нақты оқиды. Төмендегі әдеттегі есеп сұрауы негізгі кестені атап, Doris-ке индекс таңдауға мүмкіндік береді және соңғы агрегаттауды орындайды; нәтижесінде сервис пен минутқа бір жол бар.

<a id="unit-u.cd7cc51a02be4524bb8b8e990d8b46d5"></a>

`otel_traces` кестесінен әр сервис пен минуттың сұраулары мен сәтсіздерін сұрап, жоспарын қараңыз:

In [ ]:
MINUTES_QUERY = """
SELECT date_trunc(end_time, 'minute') AS request_minute,
       service_name,
       COUNT(*) AS requests,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_requests
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER'
GROUP BY date_trunc(end_time, 'minute'), service_name
"""
lab.explain_scan(MINUTES_QUERY, title="The plan of the counts per service and minute");

<a id="unit-u.25ce4262abf74b0397f4fa4158369a19"></a>

**Күтілетін нәтиже**

`reads` бағаны `server_minutes_mv` болатын `otel.otel_traces` бір сканерлеуі: Doris кесте span-дарының орнына көріністің бірнеше жүз жолын оқуға сұрауды қайта жазды. Екінші кесте көріністі `chosen` деп тізеді. Қай жолмен оқыса да нәтиже бірдей; көру үшін `lab.sql(MINUTES_QUERY)` орындаңыз.

<a id="unit-u.1cca7f8967104e0c9863bf0963aa1d5f"></a>

Doris көрініс жолдарынан сұрау нәтижесін есептей алса ғана оны қолданады. Әрқайсысы көріністен бір жайтпен ерекшеленетін үш сұраудың жоспарын қараңыз: `end_time` өзіне сүзгі, `span_kind` сүзгісінің жоқтығы және көріністе сақталмайтын ең үлкен ұзақтық.

In [ ]:
for title, query in (
    ("A filter on end_time", """
SELECT date_trunc(end_time, 'minute') AS request_minute, COUNT(*) AS requests
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER' AND service_name = 'payment'
  AND end_time >= UTC_TIMESTAMP() - INTERVAL 30 MINUTE
GROUP BY date_trunc(end_time, 'minute')
"""),
    ("Every span, not only server spans", """
SELECT date_trunc(end_time, 'minute') AS request_minute, service_name, COUNT(*) AS spans
FROM otel_traces
GROUP BY date_trunc(end_time, 'minute'), service_name
"""),
    ("The maximum duration", """
SELECT date_trunc(end_time, 'minute') AS request_minute, service_name, MAX(duration) AS max_duration
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER'
GROUP BY date_trunc(end_time, 'minute'), service_name
"""),
):
    lab.explain_scan(query, title=f"{title}: the plan");

<a id="unit-u.310de2443170483c9fa33c4d2456b810"></a>

**Күтілетін нәтиже**

Үш жоспар да `otel_traces` өзін оқып, көріністі әр себебімен `not usable` деп тізеді:

- **`end_time` сүзгісі**: `Rewrite compensate predicate by view fail`. Көрініс span-ның `end_time` мәнін емес, тек минутын сақтайды, сондықтан Doris сүзгіні көрініс жолына қолдана алмайды. `date_trunc(end_time, 'minute') >= ...` сияқты көрініс өрнегінің сүзгісі қолдануға мүмкіндік берер еді; жаттығу солай істейді.
- **Әр span**: `Predicate compensate fail`. Көріністе тек сервер span-дары бар; қалғанын санай алмайды.
- **Ең үлкен ұзақтық**: `Query function roll up fail`. Көрініс ұзақтық емес, санақ пен қосынды сақтайды.

Қайта жазу құрылымдық: Doris сұрау өрнектері, сүзгілері мен агрегаттарын көрініспен сәйкестендіреді, әртүрлі жазылған өрнектердің мағынасы бір екенін дәлелдеуге тырыспайды. Тақталар мен ескертулер бір пішімде қайта-қайта орындайтын сұрауларға көрініс тиімді. Құны да бар: енді `otel_traces` әр жүктеуі көріністі де жаңартады, көрініс диск алады.

<a id="unit-u.c3e32abf424f4853a58de188c61b068c"></a>

<a id="5-compare-compression-algorithms"></a>

## 5. Сығу алгоритмдерін салыстыру

Doris дискіге жазар алдында әр беттің әр бағанын сығады. Алгоритм — кесте құрылғанда орнатылатын `compression` қасиеті. Әдепкі ZSTD көбірек сығады; LZ4 азырақ сығады, бірақ ашуға аз CPU жұмсайды. Салыстыруда қалғанының бәрі бірдей болуы керек: бірдей жолдар бірдей тәсілмен тек `compression` бойынша ерекшеленетін кестелерге жүктелуі тиіс. Келесі ұяшық `otel_logs` негізгі бағандары бар, индекссіз `logs_zstd` пен `logs_lz4` кестелерін құрып, бірдей журнал жолдарын әрқайсысына бір `INSERT` арқылы жүктейді:

In [ ]:
for algorithm in ("zstd", "lz4"):
    lab.execute(f"""
CREATE TABLE IF NOT EXISTS logs_{algorithm} (
    timestamp           DATETIME(6),
    service_name        VARCHAR(200),
    trace_id            VARCHAR(200),
    span_id             STRING,
    severity_number     INT,
    severity_text       STRING,
    body                STRING,
    resource_attributes VARIANT,
    log_attributes      VARIANT
)
DUPLICATE KEY(timestamp, service_name)
DISTRIBUTED BY RANDOM BUCKETS 1
PROPERTIES ("compression" = "{algorithm}")
""", title=f"Create logs_{algorithm}")

copy_limit = lab.sql("SELECT MAX(timestamp) - INTERVAL 30 SECOND AS copy_limit FROM otel_logs",
                     title="Copy the log rows before this time")["copy_limit"][0]
for algorithm in ("zstd", "lz4"):
    lab.execute(f"TRUNCATE TABLE logs_{algorithm}", title="Empty the table, so that the cell can be rerun")
    lab.execute(f"""
INSERT INTO logs_{algorithm}
SELECT timestamp, service_name, trace_id, span_id, severity_number, severity_text, body, resource_attributes, log_attributes
FROM otel_logs
WHERE timestamp < '{copy_limit}'
""", title=f"Copy the same log rows into logs_{algorithm}")
copied = lab.sql("SELECT COUNT(*) AS copied_rows FROM logs_zstd", title="Rows copied")["copied_rows"][0]

<a id="unit-u.333b27b4462c465f831c855458ee8c92"></a>

**Күтілетін нәтиже**

Екі `CREATE TABLE` карточкасы, көшіру шегі және бірдей ондаған мың жолды хабарлайтын екі кірістіру. Әр кестеде бір bucket және бір жүктеу бар, сондықтан жолдары бірдей орналасқан бір rowset ішінде.

<a id="unit-u.fab73bf4de9f4d15a7ad1fdbc8826495"></a>

Doris кесте өлшемін шамамен әр минут келетін BE есептерінен санайды. Келесі ұяшық екі кесте де барлық жолын хабарлағанша күтіп, өлшемдерін көрсетеді:

In [ ]:
sizes = lab.wait_for_rows(
    f"""
    SELECT TABLE_NAME AS table_name,
           TABLE_ROWS AS table_rows,
           DATA_LENGTH AS data_bytes,
           ROUND(DATA_LENGTH / 1048576, 2) AS data_mb
    FROM information_schema.tables
    WHERE TABLE_SCHEMA = 'otel'
      AND TABLE_NAME IN ('logs_zstd', 'logs_lz4')
      AND TABLE_ROWS = {copied}
      AND DATA_LENGTH > 0
    ORDER BY data_bytes
    """,
    title="Sizes of the two tables, once the BEs have reported them",
    hint="The BEs have not reported the tables' sizes yet. Rerun this cell in a minute.",
    min_rows=2,
    timeout_seconds=180,
)
print(f"LZ4 needs {sizes['data_bytes'].max() / sizes['data_bytes'].min():.2f} times the space of ZSTD for the same rows.")

<a id="unit-u.13665b569fd94c3f8283f215329b4c8c"></a>

**Күтілетін нәтиже**

Бір-екі минутта `table_rows` бірдей екі жол: алдымен ең кішісі `logs_zstd`, кейін үлкенірек `logs_lz4`, және арақатынасы — курс журналдарына шамамен 1.5–1.6: бірдей жолдарға LZ4 шамамен 50%–60% артық диск қажет етеді. Қатынас журналы жолдары жиі қайталанады, оны ZSTD жақсырақ пайдаланады.

Салыстыру осы жолдар мен осы орналасуға ғана қатысты. Басқа дерек басқаша сығылады; нақты кестеде индекстер мен compaction көлемге қосылады. Жылдамдық туралы да ештеңе айтпайды: LZ4 ашуға аз CPU жұмсайды; шағын жиі оқылатын дерек кестесі сияқты сақталғаннан әлдеқайда көп оқылатын кестеге бұл маңызды. Бір рет жазылып, күндер сақталып, көбіне шағын бөліктермен оқылатын телеметрияға экспорттаушы таңдаған ZSTD диск үнемдейді. Алгоритм кесте құрылғанда бекітіледі; өзгерту үшін жаңа кесте құрып, деректі көшіріңіз.

<a id="unit-u.bbaf832cd09148cfaaebdb410be070de"></a>

<a id="6-exercise-a-query-that-the-view-answers"></a>

## 6. Жаттығу: көрініс жауап беретін сұрау

`otel_traces` бойынша соңғы 30 минуттың әр минуты үшін payment сұраулары мен сәтсіз сұрауларын қайтаратын, Doris `server_minutes_mv` арқылы жауап беретін сұрау жазыңыз. Жоспарын `lab.explain_scan(...)` арқылы тексеріңіз: `reads` — `server_minutes_mv` болуы керек. 4-бөлім қай уақыт сүзгісі көріністі қолдануға кедергі болатынын көрсетті.

In [ ]:
# Your code here

<a id="unit-u.bca0a18fc65543d9a3fc8b5326b783c8"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған. Салыстыру үшін ашыңыз немесе орындаңыз.

In [ ]:
PAYMENT_MINUTES = """
SELECT date_trunc(end_time, 'minute') AS request_minute,
       COUNT(*) AS requests,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_requests
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER'
  AND service_name = 'payment'
  AND date_trunc(end_time, 'minute') >= date_trunc(UTC_TIMESTAMP() - INTERVAL 30 MINUTE, 'minute')
GROUP BY date_trunc(end_time, 'minute')
ORDER BY request_minute
"""
lab.sql(PAYMENT_MINUTES, title="The payment service's requests per minute")
lab.explain_scan(PAYMENT_MINUTES, title="The plan of the payment service's requests per minute");

<a id="unit-u.6661ca64b7df4482a0e8e404c8060bc3"></a>

**Күтілетін нәтиже**

Payment төлем өңдеген әр минутқа бір жол, 31-ге дейін; бір-төрт сұрау, шамамен жартысы сәтсіз. Жоспар `server_minutes_mv` оқиды, `predicates` көріністің `mv_service_name`, `mv_request_minute` бағандарын атайды: `end_time` емес, минут сүзгісі Doris-ке оны көрініс жолдарына қолдануға мүмкіндік береді.

<a id="unit-u.4c91140f36b645d59f88b1920b5c32e0"></a>

<a id="you-are-done-when"></a>

## Аяқтау шарттары

- `otel_logs` анықтамасындағы уақыт қатары compaction саясаты, Group Commit баптаулары және жоқ `compression` нені білдіретінін айта аласыз;
- өлшеміңіз `body` кері индексі Doris оқымай тұрып дерлік барлық жолды алып тастайтынын, ал сол іздеу индекссіз әр хабарды оқитынын көрсетеді;
- `server_minutes_mv` — `FINISHED`, сервис пен минут санақтарының жоспары оны оқиды;
- бірдей журнал жолдарының ZSTD және LZ4 өлшемін өлшедіңіз.

<a id="clean-up"></a>

## Тазалау

Көрініс `otel_traces` әр жүктеуіне жұмыс қосады; кейінгі модульдер оны не екі журнал кестесін қолданбайды. Жою үшін `lab.execute(...)` не MySQL клиентімен `DROP MATERIALIZED VIEW IF EXISTS server_minutes_mv ON otel_traces`, `DROP TABLE IF EXISTS logs_zstd` және `DROP TABLE IF EXISTS logs_lz4` орындаңыз. 3-бөлім сеанс айнымалылары ядро қайта іске қосылғанша сақталады.

<a id="unit-u.ab1bddf39aa948a1b50fa5f141297416"></a>

<a id="next"></a>

## Әрі қарай

[6-тест: Кесте қасиеттері, материалданған көріністер және индекстер](quiz06_properties_mv_and_indexes.ipynb) файлын ашып, [7-модуль: Трассаларды талдау және Litefuse](../module07-data-analysis-litefuse/course.md) бөліміне өтіңіз.